# Distributed alignment search: find a causal basis for one variable

Distributed alignment search (DAS) asks whether a network computes a variable from a
high-level causal model, and where. It learns an orthogonal rotation of one hidden layer such
that swapping a few rotated coordinates between two inputs changes the network's output
exactly as swapping the variable changes the causal model's output. The fraction of swaps
that match is the interchange intervention accuracy (IIA). When IIA is near 1, the learned
rows are a causal basis for the variable: a direction the network reads the variable from.

Paper: Atticus Geiger, Zhengxuan Wu, Christopher Potts, Thomas Icard and Noah D. Goodman,
"Finding Alignments Between Interpretable Causal Variables and Distributed Neural
Representations", CLeaR 2024, https://arxiv.org/abs/2303.02536.

The task is the paper's hierarchical equality: four entities `w, x, y, z`, and the label is
whether `(w == x)` equals `(y == z)`. The causal model has two boolean variables,
`V1 = (w == x)` and `V2 = (y == z)`. TorchLens does three things here: a bound hook carries
gradients from the network's loss into the rotation during training; the learned rows are
then tested as a builtin `tl.subspace` edit with `tl.patch_from` on held-out pairs; and the
same edit, applied to a random direction and to the whole layer, gives the two controls.
The model is a small MLP trained in the first cell, so the notebook runs offline in under a
minute.

In [ ]:
# recipe-test: tool=torchlens expected=True
import torch
from torch import nn

import torchlens as tl

VOCAB, DIM, HIDDEN = 20, 4, 32
EMBEDDINGS = torch.randn(VOCAB, DIM, generator=torch.Generator().manual_seed(0))


def sample(n: int, generator: torch.Generator) -> torch.Tensor:
    """Entity ids ``[n, 4]`` for ``w, x, y, z``; each pair is equal half the time."""
    ids = torch.randint(0, VOCAB, (n, 4), generator=generator)
    for left, right in ((0, 1), (2, 3)):
        same = torch.rand(n, generator=generator) < 0.5
        ids[same, right] = ids[same, left]
        clash = ~same & (ids[:, right] == ids[:, left])
        ids[clash, right] = (ids[clash, left] + 1) % VOCAB
    return ids


def encode(ids: torch.Tensor) -> torch.Tensor:
    return EMBEDDINGS[ids].reshape(len(ids), 4 * DIM)


def causal_variables(ids: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """The causal model's two variables, ``V1 = (w == x)`` and ``V2 = (y == z)``."""
    return ids[:, 0] == ids[:, 1], ids[:, 2] == ids[:, 3]


class EqualityNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layer1 = nn.Linear(4 * DIM, HIDDEN)
        self.layer2 = nn.Linear(HIDDEN, HIDDEN)
        self.head = nn.Linear(HIDDEN, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(torch.relu(self.layer2(torch.relu(self.layer1(x)))))


torch.manual_seed(0)
model = EqualityNet()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
generator = torch.Generator().manual_seed(1)
for _ in range(2000):
    ids = sample(256, generator)
    v1, v2 = causal_variables(ids)
    loss = nn.functional.cross_entropy(model(encode(ids)), (v1 == v2).long())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
model.eval().requires_grad_(False)

ids = sample(4000, generator)
v1, v2 = causal_variables(ids)
accuracy = (model(encode(ids)).argmax(-1) == (v1 == v2).long()).float().mean().item()
print(f"task accuracy {accuracy:.3f}")
RESULT = accuracy > 0.99

## 1. What the causal model predicts for a swap

An interchange takes a base input and a source input and runs the base with one variable
set to its value on the source. For `V1` the causal model then outputs
`source.V1 == base.V2`. About half of these labels differ from the base's own label, so
a basis that does nothing scores about 0.5.

In [ ]:
# recipe-test: tool=torchlens expected=True
def swap_label(base: torch.Tensor, source: torch.Tensor) -> torch.Tensor:
    """The causal model's output on ``base`` with ``V1`` taken from ``source``."""
    _, base_v2 = causal_variables(base)
    source_v1, _ = causal_variables(source)
    return (source_v1 == base_v2).long()


pairs = torch.Generator().manual_seed(2)
train_base, train_source = sample(8192, pairs), sample(8192, pairs)
train_label = swap_label(train_base, train_source)
base_v1, base_v2 = causal_variables(train_base)
changed = (train_label != (base_v1 == base_v2).long()).float().mean().item()
RESULT = 0.4 < changed < 0.6

## 2. Learn the rotation through a bound hook

The rotation is an `nn.Linear` with PyTorch's orthogonal parametrization; its first `k` rows
are the candidate basis `B`. The hook replaces the base's component in the span of `B` with
the source's: `h + (h_source - h) B^T B`. `tl.when(site, hook).bind(model)` runs the model
with that hook and no capture, and the result carries gradients into the rotation, so plain
PyTorch training works. The model's own weights stay frozen. The source activations come
from one TorchLens trace of the source inputs.

In [ ]:
# recipe-test: tool=torchlens expected=True
def fit_das(site: str, k: int = 1, epochs: int = 8) -> tuple[torch.Tensor, float]:
    """Learn ``k`` orthonormal rows at ``site``; return them and their training-pair IIA."""
    base_inputs = encode(train_base)
    source_activation = tl.trace(model, encode(train_source))[site].out.detach()
    rotation = nn.utils.parametrizations.orthogonal(nn.Linear(HIDDEN, HIDDEN, bias=False))
    batch: dict[str, torch.Tensor] = {}

    def interchange(activation: torch.Tensor, *, ctx: object) -> torch.Tensor:
        basis = rotation.weight[:k]
        return activation + (batch["source"] - activation) @ basis.T @ basis

    bound = tl.when(tl.module(site), interchange).bind(model)
    optimizer = torch.optim.Adam(rotation.parameters(), lr=1e-2)
    for epoch in range(epochs):
        order = torch.randperm(len(train_base), generator=torch.Generator().manual_seed(epoch))
        for rows in order.split(256):
            batch["source"] = source_activation[rows]
            loss = nn.functional.cross_entropy(bound(base_inputs[rows]), train_label[rows])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        batch["source"] = source_activation
        matched = (bound(base_inputs).argmax(-1) == train_label).float().mean().item()
    return rotation.weight[:k].detach(), matched


basis, train_iia = fit_das("layer2", k=1)
print(f"interchange accuracy on the training pairs {train_iia:.3f}")
RESULT = train_iia > 0.95

## 3. Test the basis on held-out pairs, against two controls

The test uses TorchLens's own subspace edit, not the training hook: on a replayable trace
of the base inputs, `fork.do(tl.subspace(site, basis, origin=...), tl.patch_from(source))`
writes the source trace's component along `basis` into the base and replays the rest of the
network. `origin=` is required: it records where the basis came from in the edit's audit
record. The controls are a random direction of the same size and the whole layer
(`tl.patch_from` with no subspace), which moves `V2` along with `V1`.

In [ ]:
# recipe-test: tool=torchlens expected=True
held_out = torch.Generator().manual_seed(3)
test_base, test_source = sample(2000, held_out), sample(2000, held_out)
test_label = swap_label(test_base, test_source)
source = tl.trace(model, encode(test_source))
target = tl.trace(model, encode(test_base), capture=tl.options.CaptureOptions(replayable=True))


def interchange_accuracy(selection: object) -> float:
    fork = target.fork()
    fork.do(selection, tl.patch_from(source))
    prediction = fork[fork.output_layers[0]].out.argmax(-1)
    return (prediction == test_label).float().mean().item()


random_row = torch.randn(1, HIDDEN, generator=torch.Generator().manual_seed(4))
scores = {
    "learned basis": interchange_accuracy(
        tl.subspace(tl.module("layer2"), basis, origin="DAS on V1", method="das")
    ),
    "random direction": interchange_accuracy(
        tl.subspace(tl.module("layer2"), random_row, origin="random control")
    ),
    "whole layer": interchange_accuracy(tl.module("layer2")),
}
for name, score in scores.items():
    print(f"{name:>16}: IIA {score:.3f}")
RESULT = (
    scores["learned basis"] > 0.95 and max(scores["random direction"], scores["whole layer"]) < 0.6
)

## 4. Where does `V1` become a direction?

The same search one layer earlier finds no one-dimensional basis: IIA stays far below the
second layer's. At the first hidden layer the network has not yet reduced `w` and `x` to a
single "equal or not" coordinate; at the second it has.

In [ ]:
# recipe-test: tool=torchlens expected=True
early_basis, _ = fit_das("layer1", k=1)
early = interchange_accuracy(
    tl.subspace(tl.module("layer1"), early_basis, origin="DAS on V1", method="das")
)
print(f"layer1 IIA {early:.3f} vs layer2 IIA {scores['learned basis']:.3f}")
RESULT = early < 0.85

A rotation learned with another library (pyvene, or a LoReFT artifact read with
`torchlens.bridge.pyreft.read`) goes through the same door:
`tl.subspace(site, R[:k], origin=...)`. TorchLens applies a basis and records where it came
from; the fitting loop stays yours.